# LSTM 时间序列预测：深度学习版
> 难度标签：中级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：10_时间序列 | 源文件：LSTM_时间序列.py | 核心功能：序列窗口化、LSTM 模型、滚动预测

## 概述

用 LSTM 网络做时间序列预测。核心思想：将历史数据窗口化（如过去 30 天预测第 31 天），用 LSTM 学习时序模式。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
# --- 导入库 ---
from sklearn.metrics import mean_squared_error, mean_absolute_error

## 数学原理

### 1. LSTM 的门控机制

LSTM 通过三个门控制信息流，解决长期依赖问题：

**遗忘门**：决定丢弃哪些信息

$$f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$$

**输入门**：决定存储哪些新信息

$$i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$$

**候选记忆**：

$$\tilde{C}_t = \tanh(W_C [h_{t-1}, x_t] + b_C)$$

**记忆更新**：

$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

**输出门**：

$$o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$$

**隐藏状态**：

$$h_t = o_t \odot \tanh(C_t)$$

其中 $\sigma$ 是 sigmoid 函数，$\odot$ 是逐元素乘法。

### 2. 滑动窗口构造

将时间序列转换为监督学习问题：

$$X_i = (y_{i}, y_{i+1}, \ldots, y_{i+L-1}), \quad Y_i = (y_{i+L}, \ldots, y_{i+L+H-1})$$

- $L$：look_back（输入窗口长度）
- $H$：horizon（预测步数）

代码中 `look_back=30, horizon=5`，用过去 30 步预测未来 5 步。

### 3. MinMaxScaler 归一化

$$\hat{y}_t = \frac{y_t - y_{min}}{y_{max} - y_{min}} \in [0, 1]$$

归一化的原因：
- LSTM 的 sigmoid/tanh 输出有界，输入也需要归一化
- 避免梯度爆炸/消失
- 加速收敛

### 4. LSTM 预测模型结构

$$\text{Input}(L) \to \text{Embedding} \to \text{LSTM}(H_{dim}) \to \text{FC}(H_{dim} \to H)$$

代码中：
- 输入：$(B, L)$ 的序列
- LSTM 输出：$(B, H_{dim})$ 的最后时间步隐藏状态
- FC 输出：$(B, H)$ 的预测值

### 5. 训练损失

$$\mathcal{L} = \frac{1}{n}\sum_{i=1}^{n}\|\hat{Y}_i - Y_i\|^2 = \text{MSE}$$

通过反向传播和 Adam 优化器更新参数。

### 6. 评估指标

**RMSE**（均方根误差）：$\text{RMSE} = \sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$

**MAE**：$\text{MAE} = \frac{1}{n}\sum|y_i - \hat{y}_i|$

反归一化后计算：$y_{real} = \hat{y} \times (y_{max} - y_{min}) + y_{min}$

### 7. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `MinMaxScaler()` | $\hat{y} = (y - y_{min}) / (y_{max} - y_{min})$ |
| `create_sequences(data, look_back, horizon)` | 滑动窗口 $(X_i, Y_i)$ |
| `nn.LSTM(embed_dim, hidden_dim, batch_first=True)` | LSTM 门控单元 |
| `nn.Linear(hidden_dim, horizon)` | 输出映射 $W \in \mathbb{R}^{H \times H_{dim}}$ |
| `MSELoss()` | $\mathcal{L} = \frac{1}{n}\|\hat{Y} - Y\|^2$ |
| `scaler.inverse_transform()` | 反归一化 $y = \hat{y}(y_{max}-y_{min})+y_{min}$ |
| `DataLoader(TensorDataset(X, y), batch_size=32)` | Mini-batch 训练 |

### 1. 生成合成时间序列

运行 1. 生成合成时间序列 的代码，观察算法在该环节的行为。

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

n = 500
t = np.arange(n, dtype=np.float32)
数据 = (np.sin(2 * np.pi * t / 50) + 0.5 * np.sin(2 * np.pi * t / 25) +
        0.01 * t + np.random.normal(0, 0.2, n)).astype(np.float32)

print("=" * 55)
print("LSTM 时间序列预测")
print("=" * 55)
print(f"数据长度: {n}, 范围: [{数据.min():.3f}, {数据.max():.3f}]")

### 2. 数据预处理

运行 2. 数据预处理 的代码，观察算法在该环节的行为。

In [ ]:
# 归一化到 [0, 1]
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(数据.reshape(-1, 1)).flatten()

# 滑动窗口构造监督学习样本
def create_sequences(data, look_back, horizon):
    """滑动窗口：用过去 look_back 步预测未来 horizon 步"""
    X, y = [], []
    for i in range(len(data) - look_back - horizon + 1):
        X.append(data[i : i + look_back])
# --- 继续 ---
        y.append(data[i + look_back : i + look_back + horizon])
    return np.array(X), np.array(y)

look_back = 30   # 输入窗口
horizon = 5      # 预测步数

X, y = create_sequences(data_scaled, look_back, horizon)
print(f"滑动窗口: look_back={look_back}, horizon={horizon}")
print(f"样本数: {len(X)}, 特征维度: {X.shape}")

# 训练/测试划分（按时间顺序）
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
print(f"训练样本: {len(X_train)}, 测试样本: {len(X_test)}")

# 转为 PyTorch 张量 [samples, time_steps, features]
X_train_t = torch.FloatTensor(X_train).unsqueeze(-1)  # (N, look_back, 1)
y_train_t = torch.FloatTensor(y_train)
X_test_t = torch.FloatTensor(X_test).unsqueeze(-1)
y_test_t = torch.FloatTensor(y_test)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

### 3. LSTM 模型定义

运行 3. LSTM 模型定义 的代码，观察算法在该环节的行为。

In [ ]:
class LSTM模型(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, output_size=horizon):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=0.1)
# --- 继续 ---
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        lstm_out, _ = self.lstm(x)          # (batch, seq_len, hidden)
        last_hidden = lstm_out[:, -1, :]    # 取最后一步 (batch, hidden)
        out = self.fc(last_hidden)          # (batch, horizon)
        return out

模型 = LSTM模型(input_size=1, hidden_size=64, num_layers=2, output_size=horizon)
参数量 = sum(p.numel() for p in 模型.parameters() if p.requires_grad)
print(f"\n模型结构:")
print(f"  LSTM 层数: 2, 隐藏维度: 64")
print(f"  可训练参数: {参数量:,}")

### 4. 训练

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(模型.parameters(), lr=0.001)
epochs = 50

print(f"\n--- 训练 ({epochs} 轮) ---")
模型.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
# --- 继续 ---
        optimizer.zero_grad()
        预测 = 模型(batch_X)
        loss = criterion(预测, batch_y)
        loss.backward()
        optimizer.step()
# --- 继续 ---
        epoch_loss += loss.item() * len(batch_X)
    avg_loss = epoch_loss / len(X_train)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"  Epoch {epoch + 1:>3}/{epochs}: MSE = {avg_loss:.6f}")

### 5. 预测与评估

使用训练好的模型进行预测，观察输出结果。

In [ ]:
模型.eval()
with torch.no_grad():
    pred_scaled = 模型(X_test_t).numpy()

# 反归一化（保持二维形状）
def inverse_2d(data):
    return scaler.inverse_transform(data.reshape(-1, 1)).reshape(data.shape)

y_test_orig = inverse_2d(y_test)
pred_orig = inverse_2d(pred_scaled)

# 整体评估
rmse = np.sqrt(mean_squared_error(y_test_orig, pred_orig))
mae = mean_absolute_error(y_test_orig, pred_orig)
print(f"\n--- 测试集评估（整体） ---")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE:  {mae:.4f}")

# 逐步评估（每一步的预测精度）
print(f"\n--- 逐步预测精度 ---")
for step in range(horizon):
    rmse_step = np.sqrt(mean_squared_error(y_test_orig[:, step], pred_orig[:, step]))
    mae_step = mean_absolute_error(y_test_orig[:, step], pred_orig[:, step])
    print(f"  第 {step + 1} 步: RMSE={rmse_step:.4f}, MAE={mae_step:.4f}")

# 展示部分预测结果
print(f"\n--- 预测 vs 真实（前 5 个样本，第 1 步） ---")
print(f"{'样本':>6}  {'真实':>8}  {'预测':>8}  {'误差':>8}")
for i in range(min(5, len(y_test_orig))):
    print(f"  {i + 1:>4}   {y_test_orig[i, 0]:>8.3f}  {pred_orig[i, 0]:>8.3f}  {y_test_orig[i, 0] - pred_orig[i, 0]:>8.3f}")

### 6. 单步迭代预测（模拟自回归）

在回归任务上拟合模型，观察预测值与真实值的偏差。

In [ ]:
print(f"\n--- 自回归预测（用模型自身输出迭代 20 步） ---")
模型.eval()
初始输入 = data_scaled[-look_back:].copy()
当前窗口 = 初始输入.copy()

自回归预测 = []
with torch.no_grad():
    for _ in range(4):
        x = torch.FloatTensor(当前窗口).unsqueeze(0).unsqueeze(-1)  # (1, look_back, 1)
        out = 模型(x).numpy().flatten()
# --- 继续 ---
        自回归预测.extend(out[:5])
        # 更新窗口：移除前 5 步，加入预测值
        当前窗口 = np.concatenate([当前窗口[5:], out[:5]])

自回归值 = scaler.inverse_transform(np.array(自回归预测).reshape(-1, 1)).flatten()
print(f"{'步数':>4}  {'预测值':>8}")
for i, v in enumerate(自回归值):
    print(f"  {i + 1:>2}   {v:>8.3f}")

## 关键代码解释

```python
# 窗口化：[t-29, t-28, ..., t] -> t+1
X_windows = sliding_window(data, window_size=30)
model = nn.LSTM(input_size=1, hidden_size=50, num_layers=2)
```

## 注意事项

1. 数据归一化很重要
2. 滚动预测误差会累积
3. 简单问题上 ARIMA 可能更好

## 总结与延伸

以上代码展示了 **LSTM 时间序列** 的完整流程。

**解读要点**：
- 观察预测曲线与实际值的 **趋势一致性**
- 关注残差是否呈现随机分布（无明显模式）
- 对比不同模型的 **MAPE / RMSE** 指标

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **Transformer for Time Series**：时序 Transformer
- **Temporal Fusion Transformer**：Google 的多步预测模型
- **时序特征工程**：滞后特征、滚动统计
